## 1. 主题对齐：跨手型手内旋转泛化的核心矛盾

**用户反馈**：讨论主题确定为“跨手型手内旋转泛化”；当前处于框架/工程设计阶段，尚未进入 coding。

**分析**：

先把问题压缩成一句话：

> 目标不是“把旋转任务做出来”，而是“在不牺牲在线控制性的前提下，让策略跨 hand embodiment 迁移到 Leap family 之外”。

现有代表工作给出的边界大致如下：

| 工作 | 强项 | 关键缺口（对本题） |
|---|---|---|
| Patel & Song 2024 - GET-Zero | joint-token + 图偏置 attention，在线策略形态成熟 | 泛化主要在 Leap family 内；embodiment 编码偏弱（joint limits/几何/驱动能力表达不足）；训练范式依赖 per-expert distillation |
| Yang 等 2024 - AnyRotate | 单手型旋转任务做得深，含 sim-to-real 经验 | 不是跨手型泛化方法论文 |
| Liu 等 2025 - DexNDM | 现实差距建模（动力学）能力强 | 主要是 sim2real 方向，不直接解决跨手型表示问题 |
| Fei 等 2025 - T(R,O) Grasp | 关系建模强（edge-conditioned message/value） | 任务是抓取+扩散，不是在线手内旋转控制主干 |

因此，本题的方法不可替代性应写成：

1. **在线 joint policy 主干必须保留**（否则不适合高频控制）。
2. **embodiment 信息必须从“弱离散偏置”升级到“可学习连续结构条件”**（否则无法跨 family）。
3. **关系信息不能只决定看谁（routing），还要决定传什么（content）**，至少在指尖/接触相关边上要成立。

对应到你现有草图，最稳主线是：

- 前段：dynamic joint tokens × static embodiment tokens（sample-wise conditioning）
- 中段：relation-aware self-attention（先稳态版本，再加重版）
- 后段：local main path + relational residual head（保证控制保真）

从工程节奏看，MVP 不宜一开始把“跨 family hardest case + 全新训练范式 + 全重关系层”三件事同时上齐。更合理是先拍板“第一性验证目标”再定网络重度。

**小结**：当前最需要先收敛的，不是模块细节，而是“第一版要证明的泛化强度”。

**待确认**：第一版 MVP 你希望优先证明哪一级泛化强度（family 内、跨 family 轻量、还是跨 family 强泛化）？

## 2. 文献与实现精读：方法核提炼

**用户反馈**：先阅读 `papers/context.md` 中指定论文与实现代码，再继续讨论收敛。

**分析**：

我已完成以下必读文献与对应实现的梳理（选读项未展开）：

- **Qi 2022 - HORA**：核心是 rapid motor adaptation（从 proprio history 估计 object extrinsics），强调“在线适应”，但不解决跨手型结构建模。
- **Yang 2024 - AnyRotate**：核心是 dense tactile（contact pose + force）+ auxiliary goal（目标姿态推进），可覆盖多旋转轴与多手姿态，但默认是单手型路线。
- **Patel & Song 2024 - GET-Zero**：核心是 joint-token + graph-biased attention（SPD/parent/child bias）+ BC distillation + FK self-modeling；优势是在线控制结构成熟，短板是 embodiment 编码偏弱且泛化主要在 Leap family 内。
- **Fei 2025 - T(R,O) Grasp**：核心是 edge-conditioned value + grouped broadcast gate（代码层更明显），关系建模强、效率高，但任务形态是扩散抓取，不是在线旋转策略。
- **Liu 2025 - DexNDM**：核心是 joint-wise neural dynamics + residual action sim2real，强调用更可扩展的真实数据收集去拟合现实动力学偏差。

结合你指定代码实现，关键可迁移结论如下：

1. **GET 的主干可直接作为在线策略骨架**：
   - 实现中确实是“score bias 主导”的图编码（SPD/parent/child/edge embedding）
   - 这保证了在线推理形态与控制频率友好。

2. **TRO 的价值不在扩散本身，而在关系注入方式**：
   - 代码中体现了 grouped gate + edge-conditioned value 的组合，
   - 可迁移的是“边不仅决定看谁，也决定传什么”。

3. **DexNDM 更像第二阶段能力增强**：
   - 其 joint-wise dynamics residual 非常适合在你已有策略稳定后做 sim2real 闭环，
   - 不建议在第一版 MVP 就与跨手型方法创新同时推进。

基于“方法不可替代性”视角，可形成三档可执行路线：

| 路线 | 方法核 | 目标贡献 | 工程风险 |
|---|---|---|---|
| M0 | GET 主干 + 更强 embodiment token/edge（不引入重 gate） | 先证明跨手型泛化成立 | 低 |
| M1 | M0 + TRO 风格 grouped gate（做消融） | 证明关系建模增益机制 | 中 |
| M2 | M1 + DexNDM 风格 residual sim2real | 从泛化走向现实部署 | 高 |

**小结**：从论文叙事和工程可执行性看，第一篇最稳是 **M0 起步、M1 作为机制增强、M2 作为后续扩展**。

**待确认**：你希望第一版论文主线定在 M0、M1，还是直接追求 M2？